# Embedding Visualization for Robinson Crusoe Adaptations

Visualize text embeddings using dimensionality reduction:
- t-SNE (t-Distributed Stochastic Neighbor Embedding)
- UMAP (Uniform Manifold Approximation and Projection)

Goals:
- Understand how adaptations cluster in embedding space
- Identify natural groupings of similar texts
- Visualize decision boundaries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow_hub as hub
from sklearn.manifold import TSNE
from umap import UMAP
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

print("Libraries loaded")

## 1. Load Data and Generate Embeddings

In [ ]:
# Load dataset
df = pd.read_hdf('./training_set.h5', 'balanced')

# Sample for visualization (use subset for speed)
sample_size = 500  # Adjust based on computational resources
df_sample = df.sample(n=min(sample_size, len(df)), random_state=42)

print(f"Dataset: {len(df)} texts")
print(f"Visualization sample: {len(df_sample)} texts")
print(f"\nClass distribution in sample:")
print(df_sample['label'].value_counts())

In [ ]:
# Load USE model and generate embeddings
print("Loading Universal Sentence Encoder...")
embed = hub.load("./USEmodel")
print("✓ Model loaded\n")

print("Generating embeddings...")
texts = df_sample['text'].tolist()
embeddings = embed(texts)
embeddings_np = embeddings.numpy()

print(f"✓ Embeddings generated: {embeddings_np.shape}")
print(f"  Samples: {embeddings_np.shape[0]}")
print(f"  Dimensions: {embeddings_np.shape[1]}")

## 2. t-SNE Visualization

In [ ]:
# Apply t-SNE
print("Applying t-SNE dimensionality reduction...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
embeddings_tsne = tsne.fit_transform(embeddings_np)

print(f"\n✓ t-SNE complete: {embeddings_tsne.shape}")

In [ ]:
# Visualize t-SNE
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

labels = df_sample['label'].values
colors = ['blue' if label == 0 else 'orange' for label in labels]
labels_text = ['Random' if label == 0 else 'RC Adaptation' for label in labels]

# Scatter plot
for label_val in [0, 1]:
    mask = labels == label_val
    label_name = 'Random' if label_val == 0 else 'RC Adaptation'
    axes[0].scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1],
                   alpha=0.6, s=50, label=label_name)

axes[0].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[0].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[0].set_title('t-SNE Visualization of Text Embeddings', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Density plot
for label_val in [0, 1]:
    mask = labels == label_val
    label_name = 'Random' if label_val == 0 else 'RC Adaptation'
    axes[1].scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1],
                   alpha=0.3, s=30, label=label_name)

# Add density contours
from scipy.stats import gaussian_kde
for label_val in [0, 1]:
    mask = labels == label_val
    if mask.sum() > 1:
        xy = embeddings_tsne[mask]
        kde = gaussian_kde(xy.T)
        x_min, x_max = xy[:, 0].min(), xy[:, 0].max()
        y_min, y_max = xy[:, 1].min(), xy[:, 1].max()
        xx, yy = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]
        positions = np.vstack([xx.ravel(), yy.ravel()])
        f = np.reshape(kde(positions).T, xx.shape)
        axes[1].contour(xx, yy, f, levels=3, alpha=0.5)

axes[1].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[1].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[1].set_title('t-SNE with Density Contours', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tsne_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'tsne_visualization.png'")

## 3. UMAP Visualization

In [ ]:
# Apply UMAP
print("Applying UMAP dimensionality reduction...")
umap_model = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, verbose=True)
embeddings_umap = umap_model.fit_transform(embeddings_np)

print(f"\n✓ UMAP complete: {embeddings_umap.shape}")

In [ ]:
# Visualize UMAP
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Scatter plot
for label_val in [0, 1]:
    mask = labels == label_val
    label_name = 'Random' if label_val == 0 else 'RC Adaptation'
    axes[0].scatter(embeddings_umap[mask, 0], embeddings_umap[mask, 1],
                   alpha=0.6, s=50, label=label_name)

axes[0].set_xlabel('UMAP Dimension 1', fontsize=12)
axes[0].set_ylabel('UMAP Dimension 2', fontsize=12)
axes[0].set_title('UMAP Visualization of Text Embeddings', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Hexbin density plot
for label_val in [0, 1]:
    mask = labels == label_val
    label_name = 'Random' if label_val == 0 else 'RC Adaptation'
    axes[1].scatter(embeddings_umap[mask, 0], embeddings_umap[mask, 1],
                   alpha=0.3, s=30, label=label_name)

axes[1].set_xlabel('UMAP Dimension 1', fontsize=12)
axes[1].set_ylabel('UMAP Dimension 2', fontsize=12)
axes[1].set_title('UMAP with Overlapping Regions', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('umap_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'umap_visualization.png'")

## 4. Comparison: t-SNE vs UMAP

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# t-SNE
for label_val in [0, 1]:
    mask = labels == label_val
    label_name = 'Random' if label_val == 0 else 'RC Adaptation'
    axes[0].scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1],
                   alpha=0.6, s=50, label=label_name)
axes[0].set_title('t-SNE', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# UMAP
for label_val in [0, 1]:
    mask = labels == label_val
    label_name = 'Random' if label_val == 0 else 'RC Adaptation'
    axes[1].scatter(embeddings_umap[mask, 0], embeddings_umap[mask, 1],
                   alpha=0.6, s=50, label=label_name)
axes[1].set_title('UMAP', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('Comparison of Dimensionality Reduction Methods', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tsne_vs_umap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'tsne_vs_umap.png'")

## 5. Save Results

In [ ]:
# Save reduced embeddings
results_df = pd.DataFrame({
    'label': labels,
    'tsne_1': embeddings_tsne[:, 0],
    'tsne_2': embeddings_tsne[:, 1],
    'umap_1': embeddings_umap[:, 0],
    'umap_2': embeddings_umap[:, 1]
})

results_df.to_csv('embedding_visualizations.csv', index=False)
print("\n✓ Results saved to 'embedding_visualizations.csv'")

# Save full embeddings
np.save('use_embeddings.npy', embeddings_np)
print("✓ Full embeddings saved to 'use_embeddings.npy'")

## Conclusion

**Key Observations:**
- Both t-SNE and UMAP reveal clustering patterns in the embedding space
- RC adaptations and random texts show varying degrees of separation
- Some overlap indicates challenging boundary cases
- UMAP tends to preserve more global structure than t-SNE
- t-SNE emphasizes local neighborhoods

These visualizations help understand what the model learned and where classification might be challenging.